# LABORATORIO CALIFICADO N° 06
## Fundamentos de Gestión de Datos — Semana 12
**Docente:** Pilar Rocío Sayán Mejía | **Semestre:** 2026-I

---
## CASO: El Catálogo Perdido de EduNacional
**Protagonista:** Valeria Cruz, Data Architect
**Empresa:** EduNacional — Ministerio de Educación

**Tareas del viceministro:**
1. Construir catálogo de metadatos de 5 datasets principales
2. Documentar diccionario de datos del dataset de matrículas
3. Proponer y justificar la arquitectura de datos más adecuada

---

## PASO 1: Generación de Datasets Sintéticos de EduNacional

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import json

np.random.seed(42)

# Dataset 1: Matriculas
n_mat=500
df_mat=pd.DataFrame({
    'id_matricula':[f'MAT{i:05d}' for i in range(1,n_mat+1)],
    'id_estudiante':[f'EST{np.random.randint(1,300):05d}' for _ in range(n_mat)],
    'id_institucion':[f'IE{np.random.randint(1,50):04d}' for _ in range(n_mat)],
    'anio':np.random.choice([2024,2025,2026],n_mat),
    'nivel':np.random.choice(['Inicial','Primaria','Secundaria'],n_mat,p=[0.2,0.4,0.4]),
    'grado':np.random.randint(1,7,n_mat),
    'modalidad':np.random.choice(['Presencial','Semi-presencial','Virtual'],n_mat,p=[0.7,0.2,0.1]),
})

# Dataset 2: Docentes
df_doc=pd.DataFrame({
    'id_docente':[f'DOC{i:05d}' for i in range(1,101)],
    'id_institucion':[f'IE{np.random.randint(1,50):04d}' for _ in range(100)],
    'especialidad':np.random.choice(['Matematica','Comunicacion','Ciencias','Historia','Arte'],100),
    'anos_servicio':np.random.randint(1,35,100),
    'nivel_remunerativo':np.random.choice(['I','II','III','IV'],100),
})

print('Datasets EduNacional generados:')
print(f'  Matriculas: {len(df_mat)}')
print(f'  Docentes:   {len(df_doc)}')
display(df_mat.head())
display(df_doc.head())

## PASO 2: Catálogo de Metadatos — Complete la ficha de cada dataset

In [ ]:
catalogo = [
    {'nombre_dataset':'matriculas','responsable':'Direccion de Matricula - MINEDU',
     'fecha_creacion':'2018-03-01','frecuencia_actualizacion':'Anual',
     'num_registros':len(df_mat),'num_columnas':len(df_mat.columns),
     'fuente':'SIAGIE (Sistema de Informacion de Apoyo a la Gestion)',
     'clasificacion':'Datos abiertos','nivel_calidad':'Medio',
     'descripcion':'Registros de matricula escolar por institucion, nivel y anio academico'},
    {'nombre_dataset':'docentes','responsable':'Direccion de Docentes - MINEDU',
     'fecha_creacion':'2015-01-01','frecuencia_actualizacion':'Semestral',
     'num_registros':len(df_doc),'num_columnas':len(df_doc.columns),
     'fuente':'NEXUS (Sistema de Personal)',
     'clasificacion':'Uso interno','nivel_calidad':'Alto',
     'descripcion':'Informacion de docentes activos con especialidad y escala remunerativa'},
]
df_catalogo=pd.DataFrame(catalogo)
print('CATALOGO DE METADATOS EDUNACIONAL:')
display(df_catalogo.T)

## PASO 3: Diccionario de Datos — Dataset Matrículas

In [ ]:
diccionario = [
    {'campo':'id_matricula','tipo':'TEXT','descripcion':'Identificador unico de matricula','valores_validos':'MATxxxxx (formato alfanumerico)','nulos_permitidos':'No','responsable':'Jefe de Matricula','ejemplo':'MAT00001'},
    {'campo':'id_estudiante','tipo':'TEXT','descripcion':'Identificador del estudiante en el sistema SIAGIE','valores_validos':'ESTxxxxx','nulos_permitidos':'No','responsable':'Area de Registro','ejemplo':'EST00042'},
    {'campo':'id_institucion','tipo':'TEXT','descripcion':'Codigo de la institucion educativa','valores_validos':'IExxxx (codigo DRE)','nulos_permitidos':'No','responsable':'Area de Instituciones','ejemplo':'IE0023'},
    {'campo':'anio','tipo':'INTEGER','descripcion':'Anio academico de la matricula','valores_validos':'2015-2030','nulos_permitidos':'No','responsable':'Coordinacion Academica','ejemplo':'2026'},
    {'campo':'nivel','tipo':'TEXT','descripcion':'Nivel educativo del estudiante','valores_validos':'Inicial, Primaria, Secundaria','nulos_permitidos':'No','responsable':'Coordinacion Academica','ejemplo':'Primaria'},
    {'campo':'grado','tipo':'INTEGER','descripcion':'Grado o anno de estudios','valores_validos':'1-6 (Primaria), 1-5 (Secundaria)','nulos_permitidos':'No','responsable':'Jefe de Matricula','ejemplo':'3'},
    {'campo':'modalidad','tipo':'TEXT','descripcion':'Modalidad de estudio','valores_validos':'Presencial, Semi-presencial, Virtual','nulos_permitidos':'No','responsable':'Direccion Pedagogica','ejemplo':'Presencial'},
]
df_dic=pd.DataFrame(diccionario)
print('DICCIONARIO DE DATOS -- Dataset Matriculas:')
display(df_dic)

## PASO 4: Linaje de Datos

In [ ]:
linaje = {
    'sistema_origen': 'SIAGIE (Sistema de Gestion Escolar)',
    'proceso_1': {'descripcion':'Registro de matricula por IE','transformacion':'Validacion de DNI via RENIEC','responsable':'Director de IE'},
    'proceso_2': {'descripcion':'Consolidacion DRE','transformacion':'Agregacion por region y nivel','responsable':'DRE (Direccion Regional)'},
    'proceso_3': {'descripcion':'Ingesta MINEDU','transformacion':'ETL nocturno, limpieza y estandarizacion','responsable':'Equipo Data MINEDU'},
    'proceso_4': {'descripcion':'Data Warehouse MINEDU','transformacion':'Modelo dimensional, tablas de hechos y dimensiones','responsable':'Data Architect'},
    'sistema_destino': 'Reportes del Viceministro y portal estadistico',
}
print('LINAJE DE DATOS -- Sistema Matricula EduNacional:')
print(json.dumps(linaje, ensure_ascii=False, indent=2))
print('\nFlujo: SIAGIE -> DRE -> ETL MINEDU -> DW -> Reportes')

## PASO 5 y 6: Propuesta de Arquitectura de Datos

In [ ]:
comparacion = {
    'criterio':['Volumen (3M estudiantes + historico)','Velocidad (batch diario + tiempo real asistencia)',
               'Variedad (CSV, JSON, XML, imagenes)','Usuarios (analistas, directivos, ciudadanos)',
               'Costo y mantenimiento'],
    'Data Lake':['Excelente','Buena (necesita herramientas extra)','Excelente','Requiere conocimiento tecnico alto','Bajo costo almacenamiento, alto costo gestion'],
    'Data Warehouse':['Buena (costoso escalar)','Solo batch','Limitada (datos estructurados)','Excelente para BI y reportes','Alto costo licencias'],
    'Data Lakehouse':['Excelente','Excelente (soporta streaming)','Excelente','Flexible para todos los perfiles','Costo medio, mantenimiento moderado'],
}
df_comp=pd.DataFrame(comparacion)
print('COMPARACION DE ARQUITECTURAS:')
display(df_comp)

print('\nARQUITECTURA RECOMENDADA: DATA LAKEHOUSE')
print('Justificacion:')
print('  1. EduNacional tiene alta variedad de datos (estructurados, semi, no estructurados)')
print('  2. Requiere tanto analisis batch (reportes ministeriales) como casi tiempo real (asistencia)')
print('  3. El Data Lakehouse combina la flexibilidad del Data Lake con la gobernanza del DW')

# Guardar catalogo en SQLite
conn=sqlite3.connect(':memory:')
df_catalogo.to_sql('catalogo_metadatos',conn,if_exists='replace',index=False)
df_dic.to_sql('diccionario_datos',conn,if_exists='replace',index=False)
print('Catalogo guardado en SQLite.')
display(pd.read_sql_query('SELECT nombre_dataset, num_registros, nivel_calidad FROM catalogo_metadatos',conn))

## ACTIVIDAD 3: Análisis Reflexivo

**A.** ¿Por qué un catálogo de metadatos reduce el tiempo de búsqueda de datos en EduNacional?

*(Su respuesta aqui)*

---

**B.** Si EduNacional recibe datos en tiempo real y en batch, ¿qué arquitectura recomendaría?

*(Su respuesta aqui)*

---

**C.** ¿Diferencia entre metadatos técnicos y de negocio? Ejemplo de cada uno con EduNacional.

*(Su respuesta aqui)*

---

**D.** Si el sistema de matrículas cambia su estructura, ¿cómo afecta el linaje de datos?

*(Su respuesta aqui)*

## CONCLUSIONES

1. *(Conclusión 1)*

2. *(Conclusión 2)*

3. *(Conclusión 3)*

---
**Docente:** Pilar Rocío Sayán Mejía | **FGD 2026-I** | **LAB-C6**